# 03 — Đọc metric và chọn artifact bằng validation

Notebook này **không tự chạy evaluation**. Nó đọc report đã có, hiển thị bảng/đồ thị và chuẩn bị đúng CLI dùng chung với repository. Metric detector, classification và retrieval được giữ tách biệt. AXE closed-set gallery evaluation nằm trong `src.evaluation.gallery`; CLI lịch sử chỉ forward vào module đó. Validation là mặc định; test chỉ chạy khi bật cờ riêng sau khi đã chốt model bằng validation.

## Khởi chạy

Chạy từ thư mục gốc của repository trong PowerShell:

```powershell
Set-Location D:/code/anhoa/train_embedding
./.venv/Scripts/Activate.ps1
jupyter lab notebooks/03_metrics_review.ipynb
```

Notebook không có ô cài hoặc nâng cấp package; mọi evaluation đều giữ opt-in.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT_CANDIDATES = (Path.cwd(), Path.cwd().parent)
PROJECT_ROOT = next((p.resolve() for p in ROOT_CANDIDATES if (p / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở Jupyter từ repository root hoặc thư mục notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Cùng implementation mà CLI dùng; notebook không chép lại công thức metric.
from src.config import resolve_repo_path
from src.evaluation.metrics import (
    classification_metrics_from_predictions,
    retrieval_recall_at_k,
)
from src.evaluation.gallery import audit_gallery_query_inputs
from src.retrieval.item_code import evaluate_item_code_retrieval

CONFIG = {
    'classification_run': 'outputs/axe_5sku/classification_100ep_s42',
    'detector_run': 'runs/model1_brand9_distillation/student_yolov8n_baseline_100ep_s42',
    'retrieval_report': 'artifacts/evaluations/axe5_classification_100ep_s42_axe5_demo_val_v3/retrieval_metrics.json',
    'checkpoint': 'checkpoints/axe_5sku/classification_100ep_s42/best_model.pth',
    'gallery_dir': 'artifacts/galleries/axe5_classification_100ep_s42_axe5_demo_v3',
    'mapping': 'artifacts/mappings/axe5_classification_100ep_s42_axe5_demo_mapping.v3.json',
    'dataset_dir': 'data/axe_crops',
}
RUN_AXE_GALLERY_VALIDATION = False
RUN_VALIDATION_EVALUATION = False
RUN_TEST_EVALUATION = False  # Opt-in riêng; không bật để tune/chọn checkpoint.
display(Markdown(f'**Repository:** `{PROJECT_ROOT}`'))

## Cấu hình và artifact được chọn

Bảng dưới chỉ kiểm tra sự tồn tại; nó không load checkpoint và không chạy model.

In [ ]:
selected = pd.DataFrame([
    {
        'key': key,
        'path': str(resolve_repo_path(value)),
        'exists': resolve_repo_path(value).exists(),
    }
    for key, value in CONFIG.items()
])
display(selected)

## Tóm tắt report lịch sử

Không so sánh trực tiếp mAP detector với Macro F1 classification hoặc Recall@1 retrieval: mỗi metric có task và denominator riêng.

In [ ]:
classification_summary_path = resolve_repo_path(CONFIG['classification_run']) / 'summary.json'
retrieval_summary_path = resolve_repo_path(CONFIG['retrieval_report'])
rows: list[dict[str, object]] = []

if classification_summary_path.is_file():
    payload = json.loads(classification_summary_path.read_text(encoding='utf-8'))
    rows.append({
        'workflow': 'classification',
        'status': payload.get('status'),
        'selection_rule': payload.get('checkpoint_selection'),
        'best_epoch': payload.get('best_epoch'),
        'primary_value': payload.get('best_val_macro_f1'),
    })
if retrieval_summary_path.is_file():
    payload = json.loads(retrieval_summary_path.read_text(encoding='utf-8'))
    rows.append({
        'workflow': 'retrieval_validation',
        'status': 'report_exists',
        'selection_rule': payload.get('evaluation_denominator'),
        'best_epoch': None,
        'primary_value': payload.get('macro_recall@1'),
    })
display(pd.DataFrame(rows) if rows else Markdown('Chưa tìm thấy summary ở các path đã chọn.'))

## Learning curves và bảng per-class

Cell dùng report đã lưu; không tính lại metric và không đọc test split.

In [ ]:
classification_dir = resolve_repo_path(CONFIG['classification_run'])
metrics_csv = classification_dir / 'metrics.csv'
if metrics_csv.is_file():
    metrics = pd.read_csv(metrics_csv)
    display(metrics.tail(10))
    plot_columns = [
        name for name in ('train_ce_loss', 'val_ce_loss', 'train_macro_f1', 'val_macro_f1')
        if name in metrics.columns
    ]
    if plot_columns:
        ax = metrics.plot(x='epoch', y=plot_columns, figsize=(11, 5), grid=True)
        ax.set_title('Classification metrics theo epoch')
        ax.set_ylabel('Giá trị report')
        plt.show()
else:
    display(Markdown(f'Không có `{metrics_csv}`.'))

per_class_csv = classification_dir / 'per_class_metrics.csv'
if per_class_csv.is_file():
    display(pd.read_csv(per_class_csv))

In [ ]:
detector_csv = resolve_repo_path(CONFIG['detector_run']) / 'results.csv'
if detector_csv.is_file():
    detector_metrics = pd.read_csv(detector_csv)
    detector_metrics.columns = [str(name).strip() for name in detector_metrics.columns]
    display(detector_metrics.tail(10))
    candidates = [
        name for name in detector_metrics.columns
        if name in {'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)'}
    ]
    if candidates:
        ax = detector_metrics.plot(y=candidates, figsize=(11, 5), grid=True)
        ax.set_title('Detector validation metrics theo epoch')
        ax.set_xlabel('Epoch row')
        plt.show()
else:
    display(Markdown(f'Không có `{detector_csv}`.'))

## Evaluation opt-in qua cùng CLI

`src.evaluation.gallery`/`scripts/test_axe_gallery.py` chỉ dùng train-gallery + validation-query. `evaluate_item_retrieval.py` mặc định dùng validation manifest cho item-code report. Test command nằm ở cell riêng và bắt buộc chỉ định `test.csv`. Hãy dùng output directory mới để không ghi đè report lịch sử.

In [ ]:
def run_checked(command: list[str]) -> None:
    print(subprocess.list2cmdline(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

axe_gallery_validation_command = [
    sys.executable, 'scripts/test_axe_gallery.py',
    '--checkpoint', CONFIG['checkpoint'],
    '--dataset-dir', CONFIG['dataset_dir'],
    '--query-split', 'val',
    '--output-dir', 'outputs/axe_5sku/manual_gallery_validation_review',
    '--device', 'auto',
]
if RUN_AXE_GALLERY_VALIDATION:
    run_checked(axe_gallery_validation_command)
else:
    print('RUN_AXE_GALLERY_VALIDATION=False — không chạy closed-set gallery validation.')

validation_command = [
    sys.executable, 'scripts/evaluate_item_retrieval.py',
    '--checkpoint', CONFIG['checkpoint'],
    '--gallery-dir', CONFIG['gallery_dir'],
    '--mapping', CONFIG['mapping'],
    '--dataset-dir', CONFIG['dataset_dir'],
    '--output-dir', 'artifacts/evaluations/manual_validation_review',
    '--device', 'auto',
]
if RUN_VALIDATION_EVALUATION:
    run_checked(validation_command)
else:
    print('RUN_VALIDATION_EVALUATION=False — chỉ hiển thị command validation:')
    print(subprocess.list2cmdline(validation_command))

In [ ]:
test_manifest = resolve_repo_path('data/axe_crops/manifests/test.csv')
test_command = [
    sys.executable, 'scripts/evaluate_item_retrieval.py',
    '--checkpoint', CONFIG['checkpoint'],
    '--gallery-dir', CONFIG['gallery_dir'],
    '--mapping', CONFIG['mapping'],
    '--dataset-dir', CONFIG['dataset_dir'],
    '--query-manifest', str(test_manifest),
    '--output-dir', 'artifacts/evaluations/manual_test_opt_in',
    '--device', 'auto',
]
if RUN_TEST_EVALUATION:
    if not test_manifest.is_file():
        raise FileNotFoundError(test_manifest)
    run_checked(test_command)
else:
    print('RUN_TEST_EVALUATION=False — test split không được đọc.')

## Cách diễn giải

- Classification Macro F1, detector mAP và retrieval Recall@1 không thay thế nhau.
- Retrieval denominator chỉ gồm query còn positive hợp lệ sau self/hash/family exclusions.
- Query thiếu gallery hoặc mất hết positive phải được báo riêng.
- Cosine similarity không phải xác suất.
- Không chọn checkpoint, threshold hoặc quyết định retrain bằng test split.